In [1]:
# import necessary libraries
import re
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain_community.document_loaders import UnstructuredWordDocumentLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
from pinecone import ServerlessSpec

/mnt/e/TanishProject/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# load environment variables
load_dotenv()

True

In [3]:
# initialize LLM 
model = init_chat_model("google_genai:gemini-2.5-flash")

In [4]:
# load the required document
file_path = "NexaCorp_Enterprise_Policy_Handbook_v5.2.docx"
loader = UnstructuredWordDocumentLoader(
            file_path,
            mode="elements"                             # breaks document into structured chunks based on layout
        )
docs = loader.load()

In [74]:
def is_header_footer(doc):
    text = doc.page_content.strip()
    category = doc.metadata.get("category", "")
    
    NOISE_PATTERNS = [
        r"Version\s+\d+[\.\d]*\s*\|",                       # "Version 5.2 |"
        r"Internal Restricted",                             # footer 
        r"Not for External Distribution",                   # footer 
        r"©\s*20\d{2}\s",                                   # copyright footer
        r"Enterprise Policy & Procedures Handbook",         # header 
        r"CONFIDENTIAL",                                    # header 
        r"NexaCorp Global, Inc."                            # cover page
    ]
    return any(re.search(p, text, re.IGNORECASE) for p in NOISE_PATTERNS)


# Apply filters 
docs_clean = [
    doc for doc in docs
    if not is_header_footer(doc) 
]
print(f"Before: {len(docs)}")
print(f"After: {len(docs_clean)}")

Before: 530
After: 510


In [75]:
print(docs[10])

page_content='Tel: +1 (415) 700-9000  |  hr@nexacorp.com  |  www.nexacorp.com' metadata={'source': 'NexaCorp_Enterprise_Policy_Handbook_v5.2.docx', 'category_depth': 0, 'filename': 'NexaCorp_Enterprise_Policy_Handbook_v5.2.docx', 'last_modified': '2026-04-30T05:28:12', 'page_number': 1, 'languages': ['eng'], 'filetype': 'application/vnd.openxmlformats-officedocument.wordprocessingml.document', 'parent_id': '455472029b5887c8acdc6c539fd7b95a', 'category': 'UncategorizedText', 'element_id': '96a4b1466765e57ee816307d4291b752'}


In [76]:
print(docs_clean[10])

page_content='' metadata={'source': 'NexaCorp_Enterprise_Policy_Handbook_v5.2.docx', 'languages': ['eng'], 'filename': 'NexaCorp_Enterprise_Policy_Handbook_v5.2.docx', 'filetype': 'application/vnd.openxmlformats-officedocument.wordprocessingml.document', 'category': 'PageBreak', 'element_id': '374c9b677aede328ee9f4fdefb686eb5'}


In [77]:
structured_docs = []
current_section = "General"                             # default section name

for doc in docs_clean:
    text = doc.page_content.strip()                     # page content without extra spaces
    category = doc.metadata.get("category", "")         # to get type of element like Title, Text etc

    if not text:                                        # skip empty chunks
        continue

    if category == "Title":                             # if chunk is Title update the current_section
        current_section = text
        continue   
    
    structured_docs.append({                            # add section and content to structured_docs
        "section": current_section,
        "content": text
    })

In [78]:
print(len(structured_docs))

327


In [ ]:
# text splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,      
    chunk_overlap=100,
    separators=["\n\n", "\n", "."]  
)

final_chunks = []

for item in structured_docs:
    section = item["section"]                           # extract the section and text of merged doc
    text = item["content"]

    chunks = splitter.split_text(text)                  # split text into chunks

    for chunk in chunks:
        final_chunks.append({                           # store content and section of final chunk 
            "content": chunk,
            "section": section
        })

In [80]:
# convert chunks to LangChain Documents
documents = []                                      

for chunk in final_chunks:
    documents.append(
        Document(
            page_content=chunk["content"],
            metadata={
                "section": chunk["section"]
            }
        )
    )

In [81]:
print(len(documents))

111


In [82]:
# create embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5"
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1686.43it/s]


In [83]:
# initialize pinecone
pc = Pinecone()
index_name = "nexacorp-rag"

# Create index (run once)
if index_name not in [i.name for i in pc.list_indexes()]:
    pc.create_index(
        name=index_name,
        dimension=768, 
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )

index = pc.Index(index_name)

In [84]:
# create Pinecone vector store
vectorstore = PineconeVectorStore.from_documents(
        documents=documents,
        embedding=embeddings,
        index_name="nexacorp-rag"
    )